In [1]:
from pathlib import Path
import json

import numpy as np
import plotly.graph_objects as go
from scipy.spatial import cKDTree

from node_manager.patch_generator import generate_patch
from orchestrator.patch_record import PatchRecord
from quantum_processing.hamiltonian_builder import hamiltonian_builder, phi_circle_field_local, compute_L


# RANDOM NODE GENERATION


In [2]:
# Random node cloud (no DXF, no critical-region split)
rng = np.random.default_rng(42)

n_nodes = 20
x_min, x_max = -1.0, 1.0
y_min, y_max = -1.0, 1.0

random_nodes = np.column_stack([
    rng.uniform(x_min, x_max, n_nodes),
    rng.uniform(y_min, y_max, n_nodes),
])

fig_nodes = go.Figure()
fig_nodes.add_trace(
    go.Scattergl(
        x=random_nodes[:, 0],
        y=random_nodes[:, 1],
        mode='markers',
        marker=dict(size=2, color='black', opacity=0.75),
        name='Random nodes',
    )
)
fig_nodes.update_layout(
    title=f'Random Input Nodes (N={len(random_nodes)})',
    xaxis_title='x',
    yaxis_title='y',
    template='plotly_white',
    width=950,
    height=760,
)
fig_nodes.update_yaxes(scaleanchor='x', scaleratio=1)
fig_nodes.show()


# Patch manager (all nodes)


In [3]:
# Patch generation on ALL nodes (no critical/normal segmentation)
L = 0.5
Q_max = 20
overlap_factor = 1

all_indices = np.arange(len(random_nodes), dtype=np.intp)
patches = generate_patch(
    L=L,
    nodes=random_nodes,
    Q_max=Q_max,
    overlap_factor=overlap_factor,
    cad_boundary_idx=None,
)

# Coverage check: ensure all nodes participate in at least one patch.
covered = set()
for p in patches:
    interior = np.asarray(p.get('interior_idx', []), dtype=np.intp)
    halo = np.asarray(p.get('halo_idx', []), dtype=np.intp)
    if interior.size:
        covered.update(interior.tolist())
    if halo.size:
        covered.update(halo.tolist())

uncovered = np.setdiff1d(all_indices, np.array(sorted(covered), dtype=np.intp), assume_unique=False)

if len(uncovered) > 0:
    # Add local fallback patches for uncovered nodes so the workflow truly uses all nodes.
    tree = cKDTree(random_nodes)
    unvisited = set(uncovered.tolist())
    extra_count = 0

    while unvisited:
        seed = unvisited.pop()
        k = min(Q_max, len(random_nodes))
        _, nn = tree.query(random_nodes[seed], k=k)
        nn = np.atleast_1d(nn).astype(np.intp)

        # Keep this fallback patch local and focused on uncovered nodes.
        chosen = [seed]
        for idx in nn:
            if len(chosen) >= Q_max:
                break
            if int(idx) in unvisited:
                chosen.append(int(idx))

        for idx in chosen:
            if idx in unvisited:
                unvisited.remove(idx)

        p = {
            'center': random_nodes[seed],
            'interior_idx': np.asarray(chosen, dtype=np.intp),
            'halo_idx': np.empty(0, dtype=np.intp),
            'patch_id': len(patches) + extra_count,
            'cad_boundary_idx_local': None,
        }
        patches.append(p)
        extra_count += 1

    # Recompute coverage after fallback patches.
    covered = set()
    for p in patches:
        interior = np.asarray(p.get('interior_idx', []), dtype=np.intp)
        halo = np.asarray(p.get('halo_idx', []), dtype=np.intp)
        if interior.size:
            covered.update(interior.tolist())
        if halo.size:
            covered.update(halo.tolist())

    uncovered = np.setdiff1d(all_indices, np.array(sorted(covered), dtype=np.intp), assume_unique=False)

print(f'Total patches: {len(patches)}')
print(f'Covered nodes: {len(covered)} / {len(random_nodes)}')
print(f'Uncovered nodes after fallback: {len(uncovered)}')


Total patches: 1
Covered nodes: 20 / 20
Uncovered nodes after fallback: 0


In [4]:
# Quick patch locality diagnostic (max pairwise span per patch)
patch_spans = []
for p in patches:
    interior = np.asarray(p.get('interior_idx', []), dtype=np.intp)
    halo = np.asarray(p.get('halo_idx', []), dtype=np.intp)
    idx = np.concatenate([interior, halo]) if halo.size else interior
    pts = random_nodes[idx]
    if len(pts) < 2:
        continue

    dmax = 0.0
    for i in range(len(pts)):
        d = np.linalg.norm(pts[i + 1:] - pts[i], axis=1)
        if len(d):
            dmax = max(dmax, float(d.max()))
    patch_spans.append(dmax)

if patch_spans:
    patch_spans = np.asarray(patch_spans, dtype=float)
    print(f'Patch span mean: {patch_spans.mean():.4f}')
    print(f'Patch span p95:  {np.percentile(patch_spans, 95):.4f}')
    print(f'Patch span max:  {patch_spans.max():.4f}')


Patch span mean: 2.2796
Patch span p95:  2.2796
Patch span max:  2.2796


# Patch records


In [5]:
records = []
for i, patch in enumerate(patches):
    interior = np.asarray(patch.get('interior_idx', []), dtype=np.intp)
    halo = np.asarray(patch.get('halo_idx', []), dtype=np.intp)

    all_idx = np.concatenate([interior, halo]) if halo.size else interior
    if all_idx.size == 0:
        continue

    patch_nodes = random_nodes[all_idx]
    cad_boundary = patch.get('cad_boundary_idx_local', None)

    rec = PatchRecord(
        patch_nodes=patch_nodes,
        boundary_nodes_idx=cad_boundary,
        global_indices=np.asarray(all_idx, dtype=np.intp),
        region_type='all',
    )
    records.append(rec)

print(f'Records created: {len(records)}')


Records created: 1


# Hamiltonian builder


In [6]:
for record in records:
    L_local = compute_L(record.patch_nodes)
    center = np.asarray(record.patch_nodes).mean(axis=0)
    dists = np.linalg.norm(np.asarray(record.patch_nodes) - center, axis=1)
    R = np.percentile(dists, 90)
    phi = phi_circle_field_local(record.patch_nodes, R=R)
    record.phi = phi
    band = 0.8 * R

    has_boundary = (record.boundary_nodes_idx is not None and len(record.boundary_nodes_idx) > 0)
    boundary_nodes = record.boundary_nodes_idx if has_boundary else None

    tuning_params = {
        'domain': 1,
        'spacing': 0.9,
        'sparsity': 1.9,
        'bend':0,
        'max_edge': 1.1,
        'density': 1.0,
        'angular_bins': 1.1,
        'collinearity': 1.2,
        'boundary_alignment': 0,
    }

    H, decomposition = hamiltonian_builder(
        phi=phi,
        r=record.patch_nodes,
        alpha=10,
        band=band,
        gamma=0,
        use_sparsity=True,
        N=int(0.9 * len(phi)),
        mu=0.25,
        use_repulsion=False,
        d_min=0.125,
        eta=0.8,
        use_bend=True,
        kappa=3.0,
        use_max_edge=True,
        d_max=1.2 * L_local,
        eta_max=40,
        use_density_field=True,
        density_radius=0.5 * L_local,
        gamma_density=20,
        use_angular_bins=True,
        num_angular_bins=6,
        eta_theta=20,
        use_collinearity_penalty=True,
        eta_col=20,
        use_boundary_alignment=has_boundary,
        boundary_nodes=boundary_nodes,
        beta=50.0,
        normalize=True,
        tuning_factors=tuning_params,
        return_decomposition=True,
    )

    record.hamiltonian_path = H


# QAOA


In [ ]:
from collections import deque
import os
import numpy as np

from prefect import flow, task
from prefect_dask.task_runners import DaskTaskRunner
from qiskit.quantum_info import SparsePauliOp

from quantum_processing.qaoa_iqm_pipeline import run_qaoa_iqm_batch


IQM_SERVER_URL = "https://resonance.meetiqm.com" 
IQM_QUANTUM_COMPUTER = os.environ.get('IQM_QUANTUM_COMPUTER', 'garnet')
TOKEN_JSON_PATH = 'token path'



def _hamiltonian_is_empty(H, tol=1e-12):
    if H is None:
        return True

    if isinstance(H, SparsePauliOp):
        Hs = H.simplify()
        coeffs = np.asarray(Hs.coeffs, dtype=complex)
        if coeffs.size == 0:
            return True
        if np.all(np.abs(coeffs) <= tol):
            return True
        return False

    # File path / payload cases are handled inside run_qaoa_iqm_batch.
    return False


@task(
    retries=1,
    retry_delay_seconds=10,
)
def run_qaoa_task(
    idx,
    hamiltonian_path,
    reps=1,
    init_params=None,
    shots=1000,
    n_iterations=12,
    candidates_per_iteration=8,
    iqm_server_url=IQM_SERVER_URL,
    iqm_quantum_computer=IQM_QUANTUM_COMPUTER,
    token_json_path=TOKEN_JSON_PATH,
    optimization_method="SPSA",
    optimize_on_hardware=True,
):
    # Guard task-level execution against one bad observable crashing all tasks.
    try:
        out = run_qaoa_iqm_batch(
            hamiltonian_path,
            iqm_server_url=iqm_server_url,
            quantum_computer=iqm_quantum_computer,
            token_json_path=token_json_path,
            reps=reps,
            init_params=init_params,
            shots=shots,
            n_iterations=n_iterations,
            candidates_per_iteration=candidates_per_iteration,
            optimization_method=optimization_method,
            optimize_on_hardware=optimize_on_hardware,
            return_topk=True,
            top_k=10,
        )
        if len(out) == 3:
            bitstring, energy, top_bitstrings = out
        else:
            bitstring, energy = out
            top_bitstrings = []
        return idx, bitstring, energy, top_bitstrings, None
    except ValueError as e:
        if 'Empty observable' in str(e):
            return idx, None, None, [], f'empty_observable: {e}'
        raise


@flow(
    task_runner=DaskTaskRunner(
        cluster_kwargs={
            'n_workers': 2,
            'threads_per_worker': 1,
            'processes': True,
            'memory_limit': 'auto',
            'timeout': '120s',
            'death_timeout': '240s',
        }
    )
)
def run_qaoa_parallel(records, qaoa_concurrency=1):
    if qaoa_concurrency < 1:
        raise ValueError('qaoa_concurrency must be >= 1')

    qaoa_futures = deque()
    qaoa_results = []

    skipped_empty = []
    for idx, record in enumerate(records):
        H = getattr(record, 'hamiltonian_path', None)
        if _hamiltonian_is_empty(H):
            skipped_empty.append(idx)
            continue

        qaoa_futures.append(
            run_qaoa_task.submit(
                idx,
                H,
            )
        )

        if len(qaoa_futures) >= qaoa_concurrency:
            qaoa_results.append(qaoa_futures.popleft().result())

    while qaoa_futures:
        qaoa_results.append(qaoa_futures.popleft().result())

    failed_empty_runtime = []
    for idx, bitstring, energy, top_bitstrings, err in qaoa_results:
        if err is not None or bitstring is None:
            failed_empty_runtime.append((idx, err))
            records[idx].bitstring = None
            records[idx].energy = None
            records[idx].top_bitstrings = []
            continue
        records[idx].bitstring = bitstring
        records[idx].energy = energy
        records[idx].top_bitstrings = top_bitstrings

    if skipped_empty:
        print(f'Skipped empty Hamiltonians (pre-check): {len(skipped_empty)}')
        print(f'  indices: {skipped_empty[:20]}{" ..." if len(skipped_empty) > 20 else ""}')

    if failed_empty_runtime:
        print(f'Runtime empty-observable failures: {len(failed_empty_runtime)}')
        preview = failed_empty_runtime[:10]
        for i, msg in preview:
            print(f'  idx={i} -> {msg}')
        if len(failed_empty_runtime) > len(preview):
            print(f'  ... {len(failed_empty_runtime) - len(preview)} more')

    completed = sum(1 for r in records if getattr(r, 'bitstring', None) is not None)
    print(f'QAOA completed records: {completed}/{len(records)}')

    return records


records = run_qaoa_parallel(records)


15:42:12.997 | INFO    | Flow run 'dandelion-quoll' - Beginning flow run 'dandelion-quoll' for flow 'run-qaoa-parallel'

15:42:13.001 | INFO    | Flow run 'dandelion-quoll' - View at http://127.0.0.1:4200/runs/flow-run/a1ef673a-997d-44e2-a029-9c1cca8cb2a6

15:42:13.003 | INFO    | prefect.task_runner.dask - Creating a new Dask cluster with `distributed.deploy.local.LocalCluster`

15:42:13.017 | INFO    | distributed.http.proxy - To route to workers diagnostics web server please install jupyter-server-proxy: python -m pip install jupyter-server-proxy

15:42:13.021 | INFO    | distributed.scheduler - State start

...

15:43:11.023 | INFO    | distributed.nanny - Nanny at 'tcp://127.0.0.1:45437' closed.

15:43:11.025 | INFO    | distributed.scheduler - Closing scheduler. Reason: unknown

15:43:11.026 | INFO    | distributed.scheduler - Scheduler closing all comms

15:43:11.028 | INFO    | Flow run 'dandelion-quoll' - Finished in state Failed("Flow run encountered an exception: ConnectionError: ('Connection aborted.', ConnectionResetError(104, 'Connection reset by peer'))")

ConnectionError: ('Connection aborted.', ConnectionResetError(104, 'Connection reset by peer'))

In [96]:
records


[PatchRecord(patch_nodes=array([[-0.12224312, -0.29094806],
        [ 0.28773024, -0.34834928],
        [ 0.65526234, -0.5461813 ],
        [-0.54552256, -0.62105728],
        [ 0.10916957, -0.74015699],
        [ 0.5222794 , -0.06655799],
        [ 0.64552323, -0.25908059],
        [-0.1131716 , -0.06088838],
        [ 0.2633288 ,  0.33962799],
        [-0.25840395,  0.48952431],
        [-0.09922812,  0.36609791],
        [ 0.39473606,  0.78624224],
        [ 0.71719584,  0.94139605],
        [ 0.5479121 ,  0.51617548],
        [ 0.9512447 , -0.61072258],
        [-0.74377273, -0.69142102],
        [ 0.57212861, -0.91239247],
        [-0.87236549, -0.04859015],
        [ 0.85352998,  0.93501946],
        [-0.8116453 ,  0.55676699]]), phi=array([-7.29078389e-01, -7.53165821e-01, -3.80377784e-01, -1.94409926e-01,
        -3.88427493e-01, -7.37963776e-01, -5.61114098e-01, -8.51163551e-01,
        -7.44961630e-01, -4.70020863e-01, -6.63682225e-01, -2.79512779e-01,
        -4.69107842e-03

In [97]:
import numpy as np


def _bitstring_to_str(bits):
    if bits is None:
        return None
    if isinstance(bits, str):
        return bits
    arr = np.asarray(bits, dtype=int).tolist()
    return ''.join(str(int(b)) for b in arr)


print('Top-10 sampled bitstrings per patch (absolute/global probabilities):')
missing = 0
for idx, record in enumerate(records):
    topk = getattr(record, 'top_bitstrings', None)
    if topk is None or len(topk) == 0:
        missing += 1
        continue

    patch_id = getattr(record, 'patch_id', f'patch_{idx}')
    bitstring = _bitstring_to_str(getattr(record, 'bitstring', None))
    n_qubits = len(getattr(record, 'global_indices', []))
    selected = bitstring.count('1') if bitstring is not None else 0

    print(f"\nPatch {idx} | id={patch_id} | qubits={n_qubits} | selected={selected}/{n_qubits}")
    print('rank  bitstring                prob_global    cumulative')

    cumulative = 0.0
    for rank, row in enumerate(topk[:10], start=1):
        bs = str(row.get('bitstring'))
        prob = float(row.get('probability', 0.0))
        cumulative += prob
        print(f"{rank:>4}  {bs:<22}  {prob:>11.6f}  {cumulative:>11.6f}")

if missing > 0:
    print(f"\nPatches without top-k debug data: {missing} (likely skipped/failed QAOA).")


Top-10 sampled bitstrings per patch (absolute/global probabilities):

Patch 0 | id=e710101c986e | qubits=20 | selected=7/20
rank  bitstring                prob_global    cumulative
   1  00100010000000000001       0.002000     0.002000
   2  00000000000000000000       0.001000     0.003000
   3  01100010000000000000       0.001000     0.004000
   4  00100100100000000000       0.001000     0.005000
   5  00110100001000000000       0.001000     0.006000
   6  00000100001100000000       0.001000     0.007000
   7  00100010111100000000       0.001000     0.008000
   8  10001000000010000000       0.001000     0.009000
   9  00000100001010000000       0.001000     0.010000
  10  01001001001010000000       0.001000     0.011000


# Gaussian patch merging


In [98]:
from node_manager.gaussian_patch_merger import merge_patch_results_gaussian


def _extract_raw_selected_indices(records):
    selected_parts = []
    for record in records:
        gidx = np.asarray(getattr(record, 'global_indices', []), dtype=np.intp)
        if gidx.size == 0:
            continue

        bitstring = getattr(record, 'bitstring', None)
        if bitstring is None:
            continue

        if isinstance(bitstring, str):
            bits = np.array([int(ch) for ch in bitstring], dtype=np.intp)
        else:
            bits = np.asarray(bitstring, dtype=np.intp)

        n = min(len(bits), len(gidx))
        if n == 0:
            continue

        local_selected = np.where(bits[:n] == 1)[0]
        if local_selected.size:
            selected_parts.append(gidx[:n][local_selected])

    if selected_parts:
        return np.unique(np.concatenate(selected_parts))
    return np.empty(0, dtype=np.intp)


def _build_patch_results(records):
    patch_results = []
    for record in records:
        gidx = np.asarray(getattr(record, 'global_indices', []), dtype=np.intp)
        if gidx.size == 0:
            continue

        bitstring = getattr(record, 'bitstring', None)
        if bitstring is None:
            continue

        if isinstance(bitstring, str):
            bits = np.array([int(ch) for ch in bitstring], dtype=np.intp)
        else:
            bits = np.asarray(bitstring, dtype=np.intp)

        n = min(len(bits), len(gidx))
        if n == 0:
            continue

        local_selected = np.where(bits[:n] == 1)[0].tolist()
        if len(local_selected) == 0:
            continue

        patch_results.append(
            {
                'patch_id': getattr(record, 'patch_id', None),
                'local_selected': local_selected,
                'patch_indices': gidx[:n],
            }
        )

    return patch_results


raw_selected_indices = _extract_raw_selected_indices(records)
patch_results = _build_patch_results(records)

gaussian_boundary_threshold = 0.5 * float(globals().get('L', 1.0))
merged_selected_indices = merge_patch_results_gaussian(
    patch_results=patch_results,
    nodes=np.asarray(random_nodes, dtype=float),
    boundary_threshold=gaussian_boundary_threshold,
)
merged_selected_indices = np.asarray(merged_selected_indices, dtype=np.intp)

if len(merged_selected_indices) > 0:
    selected_indices_for_downstream = merged_selected_indices
    selected_source = 'merged'
else:
    selected_indices_for_downstream = raw_selected_indices
    selected_source = 'raw_fallback'

print(f'Raw selected nodes: {len(raw_selected_indices)}')
print(f'Merged selected nodes: {len(merged_selected_indices)}')
print(f'Selected source for downstream: {selected_source}')
print(f'Gaussian boundary threshold: {gaussian_boundary_threshold}')


  Total selected nodes across patches: 7
    Merged 2 nodes (Gaussian-weighted)
  Final merged nodes: 7
Raw selected nodes: 7
Merged selected nodes: 7
Selected source for downstream: merged
Gaussian boundary threshold: 0.25


# Save


In [99]:
import json
from datetime import datetime
from pathlib import Path


def _json_safe(value):
    if value is None or isinstance(value, (str, int, float, bool)):
        return value
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, (np.integer, np.floating, np.bool_)):
        return value.item()
    if isinstance(value, np.ndarray):
        return [_json_safe(v) for v in value.tolist()]
    if isinstance(value, (list, tuple, set)):
        return [_json_safe(v) for v in value]
    if isinstance(value, dict):
        return {str(k): _json_safe(v) for k, v in value.items()}
    if hasattr(value, 'to_list'):
        try:
            return _json_safe(value.to_list())
        except Exception:
            pass
    return str(value)


def _extract_selected_nodes_from_records(records):
    selected_parts = []
    for record in records:
        gidx = np.asarray(getattr(record, 'global_indices', []), dtype=np.intp)
        if gidx.size == 0:
            continue

        bitstring = getattr(record, 'bitstring', None)
        if bitstring is None:
            continue

        if isinstance(bitstring, str):
            bits = np.array([int(ch) for ch in bitstring], dtype=np.intp)
        else:
            bits = np.asarray(bitstring, dtype=np.intp)

        n = min(len(bits), len(gidx))
        if n == 0:
            continue

        local_selected = np.where(bits[:n] == 1)[0]
        if local_selected.size:
            selected_parts.append(gidx[:n][local_selected])

    if selected_parts:
        return np.unique(np.concatenate(selected_parts)).tolist()
    return []


def _extract_patched_nodes_from_records(records):
    patched_parts = []
    for record in records:
        gidx = np.asarray(getattr(record, 'global_indices', []), dtype=np.intp)
        if gidx.size:
            patched_parts.append(gidx)

    if patched_parts:
        return np.unique(np.concatenate(patched_parts)).tolist()
    return []


def _collect_tuning_parameters():
    params = {
        'node_source': 'random_uniform',
        'n_nodes': int(len(random_nodes)),
        'x_min': float(x_min),
        'x_max': float(x_max),
        'y_min': float(y_min),
        'y_max': float(y_max),
        'L': float(L),
        'Q_max': int(Q_max),
        'overlap_factor': float(overlap_factor),
        'use_gaussian_patch_merging': True,
        'gaussian_boundary_threshold': float(globals().get('gaussian_boundary_threshold', 0.5 * float(L))),
        'selected_source': str(globals().get('selected_source', 'raw_fallback')),
        'reps': 1,
        'shots': 1000,
        'n_iterations': 12,
        'candidates_per_iteration': 8,
        'optimization_method': 'SPSA',
        'optimize_on_hardware': True,
        'iqm_server_url': str(os.environ.get('IQM_SERVER_URL', 'https://cocos.resonance.meetiqm.com')),
        'iqm_quantum_computer': str(os.environ.get('IQM_QUANTUM_COMPUTER', 'garnet')),
        'token_json_path': 'token.json',
        'qaoa_concurrency': 1,
    }
    return _json_safe(params)


timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
run_dir = Path('outputs') / f'records_random_run_{timestamp}'
run_dir.mkdir(parents=True, exist_ok=True)

records_payload = []
for record in records:
    records_payload.append(
        {
            'patch_id': _json_safe(getattr(record, 'patch_id', None)),
            'patch_nodes': _json_safe(getattr(record, 'patch_nodes', None)),
            'phi': _json_safe(getattr(record, 'phi', None)),
            'boundary_nodes_idx': _json_safe(getattr(record, 'boundary_nodes_idx', None)),
            'global_indices': _json_safe(getattr(record, 'global_indices', None)),
            'region_type': _json_safe(getattr(record, 'region_type', None)),
            'hamiltonian_path': _json_safe(getattr(record, 'hamiltonian_path', None)),
            'decomposition_path': _json_safe(getattr(record, 'decomposition_path', None)),
            'bitstring': _json_safe(getattr(record, 'bitstring', None)),
            'energy': _json_safe(getattr(record, 'energy', None)),
            'decomposition': _json_safe(getattr(record, 'decomposition', None)),
        }
    )

records_path = run_dir / 'records_state.json'
with records_path.open('w') as f:
    json.dump(records_payload, f, indent=2)

nodes_array = np.asarray(random_nodes)
all_nodes_idx = np.arange(len(nodes_array), dtype=np.intp).tolist()
patched_nodes = _extract_patched_nodes_from_records(records)
raw_selected_nodes = _extract_selected_nodes_from_records(records)
merged_selected_nodes = np.asarray(globals().get('merged_selected_indices', np.empty(0, dtype=np.intp)), dtype=np.intp).tolist()
stitched_selected_nodes = np.asarray(globals().get('stitched_selected_indices', np.empty(0, dtype=np.intp)), dtype=np.intp).tolist()
downstream_selected_nodes = np.asarray(globals().get('selected_indices_for_downstream', np.empty(0, dtype=np.intp)), dtype=np.intp).tolist()
selected_source_name = str(globals().get('selected_source', 'raw_fallback'))

if len(downstream_selected_nodes) > 0:
    selected_nodes = downstream_selected_nodes
elif len(merged_selected_nodes) > 0:
    selected_nodes = merged_selected_nodes
else:
    selected_nodes = raw_selected_nodes

selection_summary = {
    'all nodes': all_nodes_idx,
    'critical region nodes': patched_nodes,
    'selected nodes': selected_nodes,
    'selected source': selected_source_name,
    'raw selected nodes': raw_selected_nodes,
    'merged selected nodes': merged_selected_nodes,
    'stitched selected nodes': stitched_selected_nodes,
}
summary_path = run_dir / 'selected_nodes_summary.json'
with summary_path.open('w') as f:
    json.dump(selection_summary, f, indent=2)

coords_payload = {
    'coordinates_by_index': _json_safe(nodes_array[:, :2]),
}
coords_path = run_dir / 'node_coordinates.json'
with coords_path.open('w') as f:
    json.dump(coords_payload, f, indent=2)

params_path = run_dir / 'tuning_parameters.json'
with params_path.open('w') as f:
    json.dump(_collect_tuning_parameters(), f, indent=2)

print(f'Saved run outputs to {run_dir}')
print(f'  - {records_path.name}')
print(f'  - {summary_path.name}')
print(f'  - {coords_path.name}')
print(f'  - {params_path.name}')



Saved run outputs to outputs/records_random_run_20260306_033635
  - records_state.json
  - selected_nodes_summary.json
  - node_coordinates.json
  - tuning_parameters.json


# Visualise


In [112]:
nodes = np.asarray(random_nodes)
if nodes.ndim != 2 or nodes.shape[1] < 2:
    raise ValueError('random_nodes must be an (N,2+) array')

patched_parts = []
raw_selected_parts = []

for record in records:
    gidx = np.asarray(getattr(record, 'global_indices', []), dtype=np.intp)
    if gidx.size == 0:
        continue

    patched_parts.append(gidx)

    bitstring = getattr(record, 'bitstring', None)
    if bitstring is None:
        continue

    if isinstance(bitstring, str):
        bits = np.array([int(ch) for ch in bitstring], dtype=np.intp)
    else:
        bits = np.asarray(bitstring, dtype=np.intp)

    n = min(len(bits), len(gidx))
    if n == 0:
        continue

    local_selected = np.where(bits[:n] == 1)[0]
    if local_selected.size:
        raw_selected_parts.append(gidx[:n][local_selected])

patched_idx = np.unique(np.concatenate(patched_parts)) if patched_parts else np.empty(0, dtype=np.intp)
raw_selected_idx = np.unique(np.concatenate(raw_selected_parts)) if raw_selected_parts else np.empty(0, dtype=np.intp)

if 'selected_indices_for_downstream' in globals() and len(selected_indices_for_downstream) > 0:
    selected_idx = np.asarray(selected_indices_for_downstream, dtype=np.intp)
    selected_source = str(globals().get('selected_source', 'merged'))
else:
    selected_idx = raw_selected_idx
    selected_source = 'raw_fallback'

all_idx = np.arange(len(nodes), dtype=np.intp)
outside_patch_idx = np.setdiff1d(all_idx, patched_idx, assume_unique=False)
patched_not_selected_idx = np.setdiff1d(patched_idx, selected_idx, assume_unique=False)

fig = go.Figure()

fig.add_trace(
    go.Scattergl(
        x=nodes[outside_patch_idx, 0],
        y=nodes[outside_patch_idx, 1],
        mode='markers',
        name='Outside patch set',
        marker=dict(color='black', size=2, opacity=0.75),
        text=outside_patch_idx,
        hovertemplate='outside %{text}<extra></extra>',
    )
)

fig.add_trace(
    go.Scattergl(
        x=nodes[patched_not_selected_idx, 0],
        y=nodes[patched_not_selected_idx, 1],
        mode='markers',
        name='Patched not selected',
        marker=dict(color='black', size=5, opacity=0.9),
        text=patched_not_selected_idx,
        hovertemplate='patched not selected %{text}<extra></extra>',
    )
)

fig.add_trace(
    go.Scattergl(
        x=nodes[selected_idx, 0],
        y=nodes[selected_idx, 1],
        mode='markers',
        name='Selected',
        marker=dict(color='red', size=6, opacity=0.95),
        text=selected_idx,
        hovertemplate='selected %{text}<extra></extra>',
    )
)

fig.update_layout(
    title=(
        f'Node Selection ({selected_source}): selected={len(selected_idx)}, '
        f'patched_not_selected={len(patched_not_selected_idx)}, '
        f'outside_patch_set={len(outside_patch_idx)}'
    ),
    xaxis_title='x',
    yaxis_title='y',
    template='plotly_white',
    width=1000,
    height=760,
)
fig.update_yaxes(scaleanchor='x', scaleratio=1)
fig.show()

print(f'Selected ({selected_source}): {len(selected_idx)}')
print(f'Raw selected: {len(raw_selected_idx)}')
print(f'Patched not selected: {len(patched_not_selected_idx)}')
print(f'Outside patch set: {len(outside_patch_idx)}')


Selected (merged): 7
Raw selected: 7
Patched not selected: 13
Outside patch set: 0


# Mesh and Results


In [101]:
from pathlib import Path

from node_manager.mesh_builder import build_and_save_mesh


def _extract_selected_indices(records):
    selected_parts = []
    for record in records:
        gidx = np.asarray(getattr(record, 'global_indices', []), dtype=np.intp)
        if gidx.size == 0:
            continue

        bitstring = getattr(record, 'bitstring', None)
        if bitstring is None:
            continue

        if isinstance(bitstring, str):
            bits = np.array([int(ch) for ch in bitstring], dtype=np.intp)
        else:
            bits = np.asarray(bitstring, dtype=np.intp)

        n = min(len(bits), len(gidx))
        if n == 0:
            continue

        local_selected = np.where(bits[:n] == 1)[0]
        if local_selected.size:
            selected_parts.append(gidx[:n][local_selected])

    if selected_parts:
        return np.unique(np.concatenate(selected_parts))
    return np.empty(0, dtype=np.intp)


raw_selected_indices = _extract_selected_indices(records)
if 'selected_indices_for_downstream' in globals() and len(selected_indices_for_downstream) > 0:
    selected_indices = np.asarray(selected_indices_for_downstream, dtype=np.intp)
    selected_source = str(globals().get('selected_source', 'merged'))
else:
    selected_indices = raw_selected_indices
    selected_source = 'raw_fallback'

if len(selected_indices) < 3:
    raise ValueError(f'Need at least 3 selected nodes for meshing, got {len(selected_indices)}')

if 'run_dir' in globals():
    mesh_output_dir = Path(run_dir) / 'mesh_selected_only'
else:
    mesh_output_dir = Path('outputs') / 'mesh_selected_only'

mesh_info = build_and_save_mesh(
    nodes=np.asarray(random_nodes, dtype=float),
    selected_indices=selected_indices,
    output_dir=mesh_output_dir,
    polygons=None,
    smooth_iterations=5,
    boundary_node_indices=None,
    formats=('msh', 'vtk', 'obj'),
)

print(f'Meshing from selected nodes ({selected_source}): {len(selected_indices)}')
print(f'Raw selected nodes: {len(raw_selected_indices)}')
print(f'Output directory: {mesh_output_dir}')
for fp in mesh_info['files']:
    print(f'  - {fp}')
if mesh_info.get('quality'):
    print('Quality:', mesh_info['quality'])



Meshing from selected nodes (merged): 7
Raw selected nodes: 7
Output directory: outputs/records_random_run_20260306_033635/mesh_selected_only
  - outputs/records_random_run_20260306_033635/mesh_selected_only/optimised_mesh.msh
  - outputs/records_random_run_20260306_033635/mesh_selected_only/optimised_mesh.vtk
  - outputs/records_random_run_20260306_033635/mesh_selected_only/optimised_mesh.obj
Quality: {'n_nodes': 7, 'n_elements': 7, 'min_angle': 17.233916962894266, 'mean_min_angle': 39.25875303576338, 'max_aspect_ratio': 2.1239358391051097, 'mean_aspect_ratio': 1.553040579330251, 'mean_skewness': 0.34568744940394375, 'max_skewness': 0.7127680506184289}


In [102]:
# Mesh quality table
quality = mesh_info.get('quality', {})
if not quality:
    raise ValueError('No mesh quality data found. Run the mesh build cell first.')

quality_keys = list(quality.keys())
quality_vals = [quality[k] for k in quality_keys]
quality_vals_fmt = [f'{v:.6g}' if isinstance(v, (int, float, np.floating)) else str(v) for v in quality_vals]

fig_q = go.Figure(
    data=[
        go.Table(
            header=dict(values=['Metric', 'Value'], fill_color='#1f2937', font=dict(color='white')),
            cells=dict(values=[quality_keys, quality_vals_fmt], fill_color='#f9fafb'),
        )
    ]
)
fig_q.update_layout(title='Mesh Quality Summary', width=650, height=420)
fig_q.show()


In [103]:
# Mesh visualization (triangles + nodes)
mesh_nodes = np.asarray(mesh_info['nodes'], dtype=float)
triangles = np.asarray(mesh_info['triangles'], dtype=np.intp)

if len(triangles) == 0:
    raise ValueError('No triangles in mesh.')

edge_x, edge_y = [], []
for t in triangles:
    pts = mesh_nodes[t]
    loop = np.vstack([pts, pts[0]])
    edge_x.extend(loop[:, 0].tolist() + [None])
    edge_y.extend(loop[:, 1].tolist() + [None])

fig_mesh = go.Figure()
fig_mesh.add_trace(
    go.Scattergl(
        x=edge_x,
        y=edge_y,
        mode='lines',
        name='Mesh edges',
        line=dict(color='rgba(0,0,0,0.35)', width=1),
    )
)
fig_mesh.add_trace(
    go.Scattergl(
        x=mesh_nodes[:, 0],
        y=mesh_nodes[:, 1],
        mode='markers',
        name='Mesh nodes',
        marker=dict(color='black', size=3, opacity=0.8),
    )
)
fig_mesh.update_layout(
    title=f'Generated Mesh (nodes={len(mesh_nodes)}, triangles={len(triangles)})',
    xaxis_title='x',
    yaxis_title='y',
    template='plotly_white',
    width=980,
    height=760,
)
fig_mesh.update_yaxes(scaleanchor='x', scaleratio=1)
fig_mesh.show()


In [109]:
# Mesh post-filter: remove overlap-prone / redundant outer triangles before heatmap
import plotly.graph_objects as go
from scipy.spatial import ConvexHull
from shapely.geometry import Polygon
from node_manager.mesh_builder import mesh_quality_summary

if 'mesh_info_prefilter' not in globals():
    mesh_info_prefilter = {
        'nodes': np.asarray(mesh_info['nodes'], dtype=float).copy(),
        'triangles': np.asarray(mesh_info['triangles'], dtype=np.intp).copy(),
        'global_idx': np.asarray(mesh_info.get('global_idx', []), dtype=np.intp).copy(),
        'quality': dict(mesh_info.get('quality', {})),
    }

base_nodes = np.asarray(mesh_info_prefilter['nodes'], dtype=float)
base_triangles = np.asarray(mesh_info_prefilter['triangles'], dtype=np.intp)
base_global_idx = np.asarray(mesh_info_prefilter.get('global_idx', []), dtype=np.intp)

if len(base_triangles) == 0:
    raise ValueError('No triangles available for post-filter.')


def _triangle_metrics(nodes, triangles):
    p = nodes[triangles]

    angles = np.zeros((len(triangles), 3), dtype=float)
    for i in range(3):
        v1 = p[:, (i + 1) % 3] - p[:, i]
        v2 = p[:, (i + 2) % 3] - p[:, i]
        cos_a = np.sum(v1 * v2, axis=1) / (
            np.linalg.norm(v1, axis=1) * np.linalg.norm(v2, axis=1) + 1e-30
        )
        angles[:, i] = np.degrees(np.arccos(np.clip(cos_a, -1, 1)))

    theta_min = angles.min(axis=1)
    theta_max = angles.max(axis=1)
    skew = np.maximum((theta_max - 60.0) / 120.0, (60.0 - theta_min) / 60.0)

    edges = np.zeros((len(triangles), 3), dtype=float)
    for i in range(3):
        edges[:, i] = np.linalg.norm(p[:, (i + 1) % 3] - p[:, i], axis=1)

    area = 0.5 * np.abs(
        (p[:, 1, 0] - p[:, 0, 0]) * (p[:, 2, 1] - p[:, 0, 1])
        - (p[:, 1, 1] - p[:, 0, 1]) * (p[:, 2, 0] - p[:, 0, 0])
    )

    return {
        'theta_min': theta_min,
        'theta_max': theta_max,
        'skew': skew,
        'edges': edges,
        'max_edge': edges.max(axis=1),
        'area': area,
    }


metrics = _triangle_metrics(base_nodes, base_triangles)
remove_mask = np.zeros(len(base_triangles), dtype=bool)
remove_reason = np.array([''] * len(base_triangles), dtype=object)

# 1. Remove genuine triangle overlaps after smoothing, keeping the better-shaped element.
tri_polys = [Polygon(base_nodes[tri]) for tri in base_triangles]
overlap_tol = 1e-12
for i in range(len(base_triangles)):
    if remove_mask[i] or tri_polys[i].area <= overlap_tol:
        continue
    for j in range(i + 1, len(base_triangles)):
        if remove_mask[j] or tri_polys[j].area <= overlap_tol:
            continue
        inter = tri_polys[i].intersection(tri_polys[j])
        if inter.area <= overlap_tol:
            continue

        score_i = (float(metrics['skew'][i]), float(metrics['max_edge'][i]), -float(metrics['area'][i]))
        score_j = (float(metrics['skew'][j]), float(metrics['max_edge'][j]), -float(metrics['area'][j]))
        if score_i >= score_j:
            remove_mask[i] = True
            remove_reason[i] = 'triangle-overlap'
            break
        else:
            remove_mask[j] = True
            remove_reason[j] = 'triangle-overlap'

# 2. Remove long, highly skewed hull-only ear triangles that create outer chords.
try:
    hull_vertices = np.asarray(ConvexHull(base_nodes).vertices, dtype=np.intp)
except Exception:
    hull_vertices = np.array([], dtype=np.intp)

if len(hull_vertices) > 0:
    hull_only = np.all(np.isin(base_triangles, hull_vertices), axis=1)
    median_edge = float(np.median(metrics['edges']))
    bad_hull_ears = (
        hull_only
        & (metrics['skew'] > 0.58)
        & (metrics['max_edge'] > 1.10 * median_edge)
    )
    new_bad = bad_hull_ears & (~remove_mask)
    remove_mask[new_bad] = True
    remove_reason[new_bad] = 'skewed-hull-ear'

filtered_triangles = base_triangles[~remove_mask]
if len(filtered_triangles) == 0:
    raise RuntimeError('Post-filter removed every triangle. Relax the filter thresholds.')

mesh_info = dict(mesh_info)
mesh_info['triangles'] = filtered_triangles
mesh_info['quality'] = mesh_quality_summary(base_nodes, filtered_triangles)
mesh_info['post_filter'] = {
    'removed_triangle_count': int(np.sum(remove_mask)),
    'removed_triangle_local_ids': np.where(remove_mask)[0].astype(int).tolist(),
    'removed_triangle_reasons': remove_reason[remove_mask].tolist(),
    'skew_threshold': 0.58,
    'edge_factor_threshold': 1.10,
}
if len(base_global_idx) > 0:
    mesh_info['global_idx'] = base_global_idx

print('Mesh post-filter applied.')
print(f"  Triangles: {len(base_triangles)} -> {len(filtered_triangles)}")
if np.any(remove_mask):
    print(f"  Removed local triangle ids: {np.where(remove_mask)[0].astype(int).tolist()}")
    print(f"  Reasons: {remove_reason[remove_mask].tolist()}")
else:
    print('  No triangles matched the filter; mesh unchanged.')

edge_x, edge_y = [], []
for tri in filtered_triangles:
    loop = np.vstack([base_nodes[tri], base_nodes[tri][0]])
    edge_x.extend(loop[:, 0].tolist() + [None])
    edge_y.extend(loop[:, 1].tolist() + [None])

fig_mesh_filtered = go.Figure()
fig_mesh_filtered.add_trace(
    go.Scattergl(
        x=edge_x,
        y=edge_y,
        mode='lines',
        name='Filtered mesh edges',
        line=dict(color='rgba(180,20,20,0.55)', width=1.25),
    )
)
fig_mesh_filtered.add_trace(
    go.Scattergl(
        x=base_nodes[:, 0],
        y=base_nodes[:, 1],
        mode='markers',
        name='Mesh nodes',
        marker=dict(color='black', size=3, opacity=0.8),
    )
)
fig_mesh_filtered.update_layout(
    title=f'Post-Filtered Mesh (nodes={len(base_nodes)}, triangles={len(filtered_triangles)})',
    xaxis_title='x',
    yaxis_title='y',
    template='plotly_white',
    width=980,
    height=760,
)
fig_mesh_filtered.update_yaxes(scaleanchor='x', scaleratio=1)
fig_mesh_filtered.show()


Mesh post-filter applied.
  Triangles: 7 -> 6
  Removed local triangle ids: [5]
  Reasons: ['triangle-overlap']


In [111]:
# Triangle-quality heatmap (2D element fill by skewness)
from plotly.colors import sample_colorscale

mesh_nodes = np.asarray(mesh_info['nodes'], dtype=float)
triangles = np.asarray(mesh_info['triangles'], dtype=np.intp)

if len(triangles) == 0:
    raise ValueError('No triangles in mesh.')

p = mesh_nodes[triangles]

angles = np.zeros((len(triangles), 3), dtype=float)
for i in range(3):
    v1 = p[:, (i + 1) % 3] - p[:, i]
    v2 = p[:, (i + 2) % 3] - p[:, i]
    cos_a = np.sum(v1 * v2, axis=1) / (
        np.linalg.norm(v1, axis=1) * np.linalg.norm(v2, axis=1) + 1e-30
    )
    angles[:, i] = np.degrees(np.arccos(np.clip(cos_a, -1, 1)))

theta_min = angles.min(axis=1)
theta_max = angles.max(axis=1)
tri_skewness = np.maximum((theta_max - 60.0) / 120.0, (60.0 - theta_min) / 60.0)

# Fixed skewness scale: 0 = equilateral, 1 = degenerate
cmin = 0.0
cmax = 1.0
denom = 1.0

fig_hm = go.Figure()

for tri, sk in zip(triangles, tri_skewness):
    pts = mesh_nodes[tri]
    loop = np.vstack([pts, pts[0]])
    t = float(np.clip((float(sk) - cmin) / denom, 0.0, 1.0))
    fill_col = sample_colorscale('Turbo', [t])[0]

    fig_hm.add_trace(
        go.Scatter(
            x=loop[:, 0],
            y=loop[:, 1],
            mode='lines',
            line=dict(color='rgba(0,0,0,0.22)', width=1),
            fill='toself',
            fillcolor=fill_col,
            hovertemplate=f'Skew={float(sk):.4f}<extra></extra>',
            showlegend=False,
        )
    )

# Invisible centroid markers only to render colorbar.
centroids = mesh_nodes[triangles].mean(axis=1)
fig_hm.add_trace(
    go.Scatter(
        x=centroids[:, 0],
        y=centroids[:, 1],
        mode='markers',
        marker=dict(
            size=0.1,
            color=tri_skewness,
            colorscale='Turbo',
            cmin=cmin,
            cmax=cmax,
            colorbar=dict(title='Skewness (0=equilateral, 1=degenerate)'),
            showscale=True,
            opacity=0.0,
        ),
        hoverinfo='skip',
        showlegend=False,
    )
)

fig_hm.update_layout(
    title='Triangle Quality Heatmap (Element Fill by Skewness)',
    xaxis_title='x',
    yaxis_title='y',
    template='plotly_white',
    width=980,
    height=780,
    dragmode='pan',
)
fig_hm.update_yaxes(scaleanchor='x', scaleratio=1)
fig_hm.show()

print(f'Skewness: mean={tri_skewness.mean():.4f}, max={tri_skewness.max():.4f}')
print(f'Min angle: mean={theta_min.mean():.2f}°, global min={theta_min.min():.2f}°')


Skewness: mean=0.2845, max=0.4777
Min angle: mean=42.93°, global min=31.34°


In [107]:
# High-skew regions + patch-region overlay + high-region aspect-ratio stats
# + non-selected node overlay (mesh-aware alignment)
from plotly.colors import sample_colorscale
import plotly.express as px
from scipy.spatial import ConvexHull

mesh_nodes = np.asarray(mesh_info['nodes'], dtype=float)
triangles = np.asarray(mesh_info['triangles'], dtype=np.intp)
mesh_global_idx = np.asarray(mesh_info['global_idx'], dtype=np.intp)

if len(triangles) == 0:
    raise ValueError('No triangles in mesh.')

p_tri = mesh_nodes[triangles]  # (T,3,2)

# Triangle angles
angles = np.zeros((len(triangles), 3), dtype=float)
for i in range(3):
    v1 = p_tri[:, (i + 1) % 3] - p_tri[:, i]
    v2 = p_tri[:, (i + 2) % 3] - p_tri[:, i]
    cos_a = np.sum(v1 * v2, axis=1) / (
        np.linalg.norm(v1, axis=1) * np.linalg.norm(v2, axis=1) + 1e-30
    )
    angles[:, i] = np.degrees(np.arccos(np.clip(cos_a, -1, 1)))

theta_min = angles.min(axis=1)
theta_max = angles.max(axis=1)
tri_skewness = np.maximum((theta_max - 60.0) / 120.0, (60.0 - theta_min) / 60.0)

# Triangle aspect ratio
edges = np.zeros((len(triangles), 3), dtype=float)
for i in range(3):
    edges[:, i] = np.linalg.norm(p_tri[:, (i + 1) % 3] - p_tri[:, i], axis=1)
aspect_ratio = edges.max(axis=1) / (edges.min(axis=1) + 1e-30)

# High-skew subset
skew_abs_threshold = 0.50
skew_percentile = 95
skew_pct_threshold = float(np.percentile(tri_skewness, skew_percentile))
skew_cutoff = max(skew_abs_threshold, skew_pct_threshold)

count_above_threshold = int(np.sum(tri_skewness >= skew_cutoff))
high_mask = tri_skewness >= skew_cutoff
if not np.any(high_mask):
    top_k = min(10, len(triangles))
    idx_top = np.argsort(tri_skewness)[-top_k:]
    high_mask[idx_top] = True

high_triangles = triangles[high_mask]
high_skew = tri_skewness[high_mask]
high_min_angle = theta_min[high_mask]
high_ar = aspect_ratio[high_mask]

# Global nodes participating in high-skew triangles
high_local_nodes = np.unique(high_triangles.reshape(-1))
high_global_nodes = set(mesh_global_idx[high_local_nodes].astype(int).tolist())

# Selected global indices used downstream (merged if present, raw fallback)
if 'selected_indices_for_downstream' in globals() and len(selected_indices_for_downstream) > 0:
    selected_global = np.asarray(selected_indices_for_downstream, dtype=np.intp)
else:
    selected_parts = []
    for rec in records:
        gidx = np.asarray(getattr(rec, 'global_indices', []), dtype=np.intp)
        if gidx.size == 0:
            continue
        bitstring = getattr(rec, 'bitstring', None)
        if bitstring is None:
            continue
        if isinstance(bitstring, str):
            bits = np.array([int(ch) for ch in bitstring], dtype=np.intp)
        else:
            bits = np.asarray(bitstring, dtype=np.intp)
        n = min(len(bits), len(gidx))
        if n == 0:
            continue
        local_selected = np.where(bits[:n] == 1)[0]
        if local_selected.size:
            selected_parts.append(gidx[:n][local_selected])
    selected_global = np.unique(np.concatenate(selected_parts)) if selected_parts else np.empty(0, dtype=np.intp)

selected_global_set = set(selected_global.astype(int).tolist())

# Select patches that overlap high-skew nodes
high_patches = []
for rec in records:
    gidx = np.asarray(getattr(rec, 'global_indices', []), dtype=np.intp)
    if gidx.size == 0:
        continue
    if any(int(g) in high_global_nodes for g in gidx):
        high_patches.append(gidx)

# Mesh-aware coordinate mapping: use smoothed mesh coords when available.
global_to_local = {int(g): i for i, g in enumerate(mesh_global_idx)}


def _coords_from_global(global_ids):
    out = []
    for gid in np.asarray(global_ids, dtype=np.intp):
        li = global_to_local.get(int(gid))
        if li is not None:
            out.append(mesh_nodes[li])
        else:
            out.append(np.asarray(random_nodes[int(gid)], dtype=float))
    if not out:
        return np.empty((0, 2), dtype=float)
    return np.asarray(out, dtype=float)

# Build patch boundaries (convex hull proxy) for overlay
patch_boundaries = []
high_patch_node_set = set()
for gidx in high_patches:
    high_patch_node_set.update(np.asarray(gidx, dtype=np.intp).astype(int).tolist())

    pts = _coords_from_global(gidx)
    if len(pts) < 3:
        continue
    try:
        hull = ConvexHull(pts)
        boundary = pts[hull.vertices]
    except Exception:
        center = pts.mean(axis=0)
        ang = np.arctan2(pts[:, 1] - center[1], pts[:, 0] - center[0])
        boundary = pts[np.argsort(ang)]

    if len(boundary) >= 3:
        boundary = np.vstack([boundary, boundary[0]])
        patch_boundaries.append(boundary)

# Non-selected nodes in high-region patches
non_selected_global = np.array(sorted(high_patch_node_set - selected_global_set), dtype=np.intp)


non_selected_nodes = _coords_from_global(non_selected_global)

# Fixed skewness scale: 0 = equilateral, 1 = degenerate
cmin = 0.0
cmax = 1.0
denom = 1.0

fig_high = go.Figure()

# Mesh context
edge_x, edge_y = [], []
for t in triangles:
    pts = mesh_nodes[t]
    loop = np.vstack([pts, pts[0]])
    edge_x.extend(loop[:, 0].tolist() + [None])
    edge_y.extend(loop[:, 1].tolist() + [None])

fig_high.add_trace(
    go.Scatter(
        x=edge_x,
        y=edge_y,
        mode='lines',
        name='Mesh context',
        line=dict(color='rgba(120,120,120,0.22)', width=1),
        hoverinfo='skip',
    )
)

# Filled high-skew triangles
for tri, sk in zip(high_triangles, high_skew):
    pts = mesh_nodes[tri]
    loop = np.vstack([pts, pts[0]])
    t = float(np.clip((float(sk) - cmin) / denom, 0.0, 1.0))
    fill_col = sample_colorscale('Reds', [t])[0]
    fig_high.add_trace(
        go.Scatter(
            x=loop[:, 0],
            y=loop[:, 1],
            mode='lines',
            line=dict(color='rgba(120,20,20,0.6)', width=1),
            fill='toself',
            fillcolor=fill_col,
            hovertemplate=f'Skew={float(sk):.4f}<extra></extra>',
            showlegend=False,
        )
    )

# Non-selected nodes (aligned via mesh-aware mapping)
if len(non_selected_nodes) > 0:
    fig_high.add_trace(
        go.Scattergl(
            x=non_selected_nodes[:, 0],
            y=non_selected_nodes[:, 1],
            mode='markers',
            name='Non-selected nodes',
            marker=dict(color='blue', size=2, opacity=0.8),
        )
    )

# Overlay boundaries of patches that contain high-skew nodes
palette = px.colors.qualitative.Alphabet + px.colors.qualitative.Dark24 + px.colors.qualitative.Light24
for i, boundary in enumerate(patch_boundaries):
    fig_high.add_trace(
        go.Scatter(
            x=boundary[:, 0],
            y=boundary[:, 1],
            mode='lines',
            line=dict(color=palette[i % len(palette)], width=1.8),
            name='High-region patch boundaries',
            legendgroup='high_region_boundaries',
            showlegend=(i == 0),
            hoverinfo='skip',
            opacity=0.9,
        )
    )

# Colorbar host
high_centroids = mesh_nodes[high_triangles].mean(axis=1)
fig_high.add_trace(
    go.Scatter(
        x=high_centroids[:, 0],
        y=high_centroids[:, 1],
        mode='markers',
        marker=dict(
            size=0.1,
            color=high_skew,
            colorscale='Reds',
            cmin=cmin,
            cmax=cmax,
            colorbar=dict(title='High skewness', x=1.02, y=0.5, len=0.78, thickness=18),
            showscale=True,
            opacity=0.0,
        ),
        hoverinfo='skip',
        showlegend=False,
    )
)

fig_high.update_layout(
    title=(
        f'High-Skew Region Diagnostics | high_triangles={len(high_triangles)}, '
        f'high_region_patches={len(patch_boundaries)}, non_selected={len(non_selected_nodes)}'
    ),
    xaxis_title='x',
    yaxis_title='y',
    template='plotly_white',
    width=1100,
    height=820,
    dragmode='pan',
    margin=dict(l=70, r=180, t=90, b=60),
    legend=dict(
        x=0.01,
        y=0.99,
        xanchor='left',
        yanchor='top',
        bgcolor='rgba(255,255,255,0.85)',
        bordercolor='rgba(0,0,0,0.2)',
        borderwidth=1,
    ),
)
fig_high.update_yaxes(scaleanchor='x', scaleratio=1)
fig_high.show()

print(f'Skewness cutoff: {skew_cutoff:.4f} (abs={skew_abs_threshold}, p{skew_percentile}={skew_pct_threshold:.4f})')
print(f'Count above threshold: {count_above_threshold} / {len(triangles)} ({100.0*count_above_threshold/max(1,len(triangles)):.2f}%)')
print(f'Plotted high-skew triangles: {len(high_triangles)} / {len(triangles)}')
print(f'High-region patches: {len(patch_boundaries)}')
print(f'Non-selected nodes (high-region patches): {len(non_selected_nodes)}')
print(f'High-region min angle: min={high_min_angle.min():.2f}°, mean={high_min_angle.mean():.2f}°')
print(f'High-region aspect ratio: max={high_ar.max():.2f}, mean={high_ar.mean():.2f}, p95={np.percentile(high_ar,95):.2f}')


Skewness cutoff: 0.6423 (abs=0.5, p95=0.6423)
Count above threshold: 1 / 7 (14.29%)
Plotted high-skew triangles: 1 / 7
High-region patches: 1
Non-selected nodes (high-region patches): 13
High-region min angle: min=17.23°, mean=17.23°
High-region aspect ratio: max=2.12, mean=2.12, p95=2.12


In [108]:
# Classical post-fix (after diagnostics): Laplacian smoothing in seam/boundary band
from scipy.spatial import ConvexHull
from node_manager.mesh_builder import mesh_quality_summary


def _laplacian_smooth_subset(nodes, triangles, movable_mask, iterations=10, weight=0.22):
    pts = np.asarray(nodes, dtype=float).copy()
    tri = np.asarray(triangles, dtype=np.intp)
    n = len(pts)

    if n == 0 or len(tri) == 0:
        return pts

    adj = [set() for _ in range(n)]
    for t in tri:
        a, b, c = int(t[0]), int(t[1]), int(t[2])
        adj[a].update((b, c))
        adj[b].update((a, c))
        adj[c].update((a, b))

    for _ in range(int(iterations)):
        new_pts = pts.copy()
        for i in np.where(movable_mask)[0]:
            if len(adj[i]) == 0:
                continue
            nbr = np.asarray(list(adj[i]), dtype=np.intp)
            centroid = pts[nbr].mean(axis=0)
            new_pts[i] = pts[i] + float(weight) * (centroid - pts[i])
        pts = new_pts

    return pts


mesh_nodes = np.asarray(mesh_info['nodes'], dtype=float)
triangles = np.asarray(mesh_info['triangles'], dtype=np.intp)
global_idx = np.asarray(mesh_info['global_idx'], dtype=np.intp)

if len(triangles) == 0:
    raise ValueError('No triangles available for post-fix.')

q_before = mesh_quality_summary(mesh_nodes, triangles)

# Target only seam + outer boundary band.
xmin, xmax = float(mesh_nodes[:, 0].min()), float(mesh_nodes[:, 0].max())
ymin, ymax = float(mesh_nodes[:, 1].min()), float(mesh_nodes[:, 1].max())
band_depth = 1.25 * float(globals().get('L', 1.0))

edge_dist = np.minimum.reduce([
    mesh_nodes[:, 0] - xmin,
    xmax - mesh_nodes[:, 0],
    mesh_nodes[:, 1] - ymin,
    ymax - mesh_nodes[:, 1],
])
boundary_band = edge_dist <= band_depth

roi = boundary_band

# Keep convex hull fixed to avoid shrinking/distorting outer contour.
fixed = np.zeros(len(mesh_nodes), dtype=bool)
if len(mesh_nodes) >= 3:
    try:
        hull = ConvexHull(mesh_nodes)
        fixed[hull.vertices] = True
    except Exception:
        pass

movable = roi & (~fixed)

postfix_nodes = _laplacian_smooth_subset(
    nodes=mesh_nodes,
    triangles=triangles,
    movable_mask=movable,
    iterations=12,
    weight=0.20,
)
q_after = mesh_quality_summary(postfix_nodes, triangles)

mesh_info_postfix = {
    'nodes': postfix_nodes,
    'triangles': triangles,
    'global_idx': global_idx,
    'quality': q_after,
    'method': 'laplacian_subset',
    'params': {
        'iterations': 12,
        'weight': 0.20,
        'band_depth': band_depth,
        'roi_nodes': int(np.sum(roi)),
        'movable_nodes': int(np.sum(movable)),
    },
}

print('Classical post-fix complete (outer-boundary Laplacian smoothing).')
print(f"  Outer-band ROI nodes: {int(np.sum(roi))}, movable: {int(np.sum(movable))}, fixed(hull): {int(np.sum(fixed))}")
print('  Quality delta:')
print(f"    mean_skewness: {q_before['mean_skewness']:.6f} -> {q_after['mean_skewness']:.6f}")
print(f"    max_skewness:  {q_before['max_skewness']:.6f} -> {q_after['max_skewness']:.6f}")
print(f"    min_angle:     {q_before['min_angle']:.6f} -> {q_after['min_angle']:.6f}")

# Optional side-by-side edge overlay
edge_x_before, edge_y_before = [], []
edge_x_after, edge_y_after = [], []
for t in triangles:
    loop_b = np.vstack([mesh_nodes[t], mesh_nodes[t][0]])
    loop_a = np.vstack([postfix_nodes[t], postfix_nodes[t][0]])
    edge_x_before.extend(loop_b[:, 0].tolist() + [None])
    edge_y_before.extend(loop_b[:, 1].tolist() + [None])
    edge_x_after.extend(loop_a[:, 0].tolist() + [None])
    edge_y_after.extend(loop_a[:, 1].tolist() + [None])

fig_post = go.Figure()
fig_post.add_trace(go.Scattergl(
    x=edge_x_before, y=edge_y_before,
    mode='lines', name='Before post-fix',
    line=dict(color='rgba(80,80,80,0.45)', width=1),
))
fig_post.add_trace(go.Scattergl(
    x=edge_x_after, y=edge_y_after,
    mode='lines', name='After post-fix',
    line=dict(color='rgba(220,30,30,0.60)', width=1),
))
fig_post.update_layout(
    title='Classical Post-Fix Comparison (Outer Region Only)',
    xaxis_title='x',
    yaxis_title='y',
    template='plotly_white',
    width=980,
    height=760,
)
fig_post.update_yaxes(scaleanchor='x', scaleratio=1)
fig_post.show()


# Post-fix skewness heatmap (after smoothing)
from plotly.colors import sample_colorscale

post_nodes = np.asarray(mesh_info_postfix['nodes'], dtype=float)
post_triangles = np.asarray(mesh_info_postfix['triangles'], dtype=np.intp)

if len(post_triangles) == 0:
    raise ValueError('No triangles in post-fix mesh.')

p_post = post_nodes[post_triangles]
angles_post = np.zeros((len(post_triangles), 3), dtype=float)
for i in range(3):
    v1 = p_post[:, (i + 1) % 3] - p_post[:, i]
    v2 = p_post[:, (i + 2) % 3] - p_post[:, i]
    cos_a = np.sum(v1 * v2, axis=1) / (
        np.linalg.norm(v1, axis=1) * np.linalg.norm(v2, axis=1) + 1e-30
    )
    angles_post[:, i] = np.degrees(np.arccos(np.clip(cos_a, -1, 1)))

theta_min_post = angles_post.min(axis=1)
theta_max_post = angles_post.max(axis=1)
tri_skew_post = np.maximum((theta_max_post - 60.0) / 120.0, (60.0 - theta_min_post) / 60.0)

# Fixed skewness scale: 0 = equilateral, 1 = degenerate
cmin_post = 0.0
cmax_post = 1.0
denom_post = 1.0

fig_hm_post = go.Figure()
for tri, sk in zip(post_triangles, tri_skew_post):
    pts = post_nodes[tri]
    loop = np.vstack([pts, pts[0]])
    t = float(np.clip((float(sk) - cmin_post) / denom_post, 0.0, 1.0))
    fill_col = sample_colorscale('Turbo', [t])[0]

    fig_hm_post.add_trace(
        go.Scatter(
            x=loop[:, 0],
            y=loop[:, 1],
            mode='lines',
            line=dict(color='rgba(0,0,0,0.22)', width=1),
            fill='toself',
            fillcolor=fill_col,
            hovertemplate=f'Skew={float(sk):.4f}<extra></extra>',
            showlegend=False,
        )
    )

centroids_post = post_nodes[post_triangles].mean(axis=1)
fig_hm_post.add_trace(
    go.Scatter(
        x=centroids_post[:, 0],
        y=centroids_post[:, 1],
        mode='markers',
        marker=dict(
            size=0.1,
            color=tri_skew_post,
            colorscale='Turbo',
            cmin=cmin_post,
            cmax=cmax_post,
            colorbar=dict(title='Skewness (0=equilateral, 1=degenerate)'),
            showscale=True,
            opacity=0.0,
        ),
        hoverinfo='skip',
        showlegend=False,
    )
)

fig_hm_post.update_layout(
    title='Triangle Quality Heatmap After Outer-Region Post-Fix (Skewness)',
    xaxis_title='x',
    yaxis_title='y',
    template='plotly_white',
    width=980,
    height=780,
    dragmode='pan',
)
fig_hm_post.update_yaxes(scaleanchor='x', scaleratio=1)
fig_hm_post.show()

print(f'Post-fix skewness: mean={tri_skew_post.mean():.4f}, max={tri_skew_post.max():.4f}')
print(f'Post-fix min angle: mean={theta_min_post.mean():.2f}°, global min={theta_min_post.min():.2f}°')




Classical post-fix complete (outer-boundary Laplacian smoothing).
  Outer-band ROI nodes: 7, movable: 1, fixed(hull): 6
  Quality delta:
    mean_skewness: 0.345687 -> 0.328144
    max_skewness:  0.712768 -> 0.712768
    min_angle:     17.233917 -> 17.233917


Post-fix skewness: mean=0.3281, max=0.7128
Post-fix min angle: mean=40.31°, global min=17.23°
